In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/hbaac-round2/sample_submission.csv
/kaggle/input/competitions/hbaac-round2/train.csv


In [2]:
import os

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))
    

/kaggle/input/competitions/hbaac-round2/sample_submission.csv
/kaggle/input/competitions/hbaac-round2/train.csv


In [3]:
train = pd.read_csv(
    '/kaggle/input/competitions/hbaac-round2/train.csv',
    low_memory=False
)

sample = pd.read_csv(
    '/kaggle/input/competitions/hbaac-round2/sample_submission.csv'
)

In [4]:
print(train.head())

print("\n====================")
print(train.columns)

print("\n====================")
print(train.info())

         Date      Stt   ItemCode  Quantity    UnitPrice  SalesAmount  \
0  2020-11-17  2000004  SKU-08063        12       242700      2184300   
1  2020-11-17  2000003  SKU-09458       600  131818,1818     79090909   
2  2020-11-18  2000007  SKU-08062         6       230000       940909   
3  2020-11-18  2000006  SKU-09458       240       270000     44181818   
4  2020-11-18  2000005  SKU-09458       240       270000     44181818   

  Unit Cost Cost Amount  
0  123559,1     1482709  
1    110000    66000000  
2    101000      606000  
3    110000    26400000  
4    110000    26400000  

Index(['Date', 'Stt', 'ItemCode', 'Quantity', 'UnitPrice', 'SalesAmount',
       'Unit Cost', 'Cost Amount'],
      dtype='object')

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 711980 entries, 0 to 711979
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype 
---  ------       --------------   ----- 
 0   Date         711980 non-null  object
 1   Stt          711980 non-null  

In [5]:
import pandas as pd
import numpy as np


# =========================
# 1. DATE -> DATETIME
# =========================

train['Date'] = pd.to_datetime(train['Date'])


# =========================
# 2. CLEAN NUMERIC COLUMNS
# =========================

numeric_cols = ['UnitPrice', 'Unit Cost', 'Cost Amount']

for col in numeric_cols:
    train[col] = (
        train[col]
        .astype(str)
        .str.replace(',', '.', regex=False)
    )

    train[col] = pd.to_numeric(train[col], errors='coerce')


# =========================
# 3. HANDLE RETURNS
# Quantity < 0 -> 0
# =========================

train['Quantity'] = train['Quantity'].clip(lower=0)


# =========================
# 4. SORT DATA
# =========================

train = train.sort_values(['ItemCode', 'Date'])


# =========================
# 5. CHECK DATA TYPES
# =========================

print(train.dtypes)


# =========================
# 6. CHECK MISSING VALUES
# =========================

print("\n===== MISSING VALUES =====")
print(train.isnull().sum())


# =========================
# 7. CREATE DAILY SALES
# Aggregate by SKU + Date
# =========================

daily_sales = (
    train
    .groupby(['Date', 'ItemCode'])['Quantity']
    .sum()
    .reset_index()
)

print("\n===== DAILY SALES =====")
print(daily_sales.head())


# =========================
# 8. BASIC INFO
# =========================

print("\n===== SHAPE =====")
print(daily_sales.shape)

print("\n===== UNIQUE SKU =====")
print(daily_sales['ItemCode'].nunique())

print("\n===== DATE RANGE =====")
print(daily_sales['Date'].min())
print(daily_sales['Date'].max())

Date           datetime64[ns]
Stt                    object
ItemCode               object
Quantity                int64
UnitPrice             float64
SalesAmount             int64
Unit Cost             float64
Cost Amount           float64
dtype: object

===== MISSING VALUES =====
Date           0
Stt            0
ItemCode       0
Quantity       0
UnitPrice      0
SalesAmount    0
Unit Cost      0
Cost Amount    0
dtype: int64

===== DAILY SALES =====
        Date   ItemCode  Quantity
0 2020-11-17  SKU-08063        12
1 2020-11-17  SKU-09458       600
2 2020-11-18  SKU-08062         6
3 2020-11-18  SKU-09458       480
4 2020-11-20  SKU-09458       240

===== SHAPE =====
(507050, 3)

===== UNIQUE SKU =====
15972

===== DATE RANGE =====
2020-11-17 00:00:00
2025-09-05 00:00:00


In [6]:
# =========================
# LAST 28 DAYS MEAN BASELINE
# =========================

# lấy 28 ngày cuối
last_date = daily_sales['Date'].max()

last_28 = daily_sales[
    daily_sales['Date'] > last_date - pd.Timedelta(days=28)
]

# mean quantity mỗi SKU
sku_mean = (
    last_28
    .groupby('ItemCode')['Quantity']
    .mean()
)

print(sku_mean.head())

ItemCode
SKU-00001     2.666667
SKU-00002     4.900000
SKU-00003    18.736842
SKU-00043     1.750000
SKU-00044     1.250000
Name: Quantity, dtype: float64


In [7]:
forecast_cols = [f'F{i}' for i in range(1, 29)]

# convert sang float trước
sample[forecast_cols] = sample[forecast_cols].astype(float)

for idx in sample.index:

    sku = sample.loc[idx, 'id'].split('_')[0]

    pred = sku_mean.get(sku, 0)

    for col in forecast_cols:
        sample.loc[idx, col] = pred

sample[forecast_cols] = sample[forecast_cols].clip(lower=0)

sample.to_csv('submission.csv', index=False)

print("submission.csv created!")

submission.csv created!
